# Huca Brugada SVM Classifier (from raw WFDB)

This notebook builds a simple classifier that distinguishes ECG records labeled as **brugada = 0** vs **brugada > 0** in `data/brugada-huca/metadata.csv`.

It reads the raw WFDB records from `data/brugada-huca/files/<patient_id>/<patient_id>`, applies a bandpass filter, extracts a small set of time-domain features for each record, and trains an SVM with class weighting to address imbalance.

In [39]:
import numpy as np
import pandas as pd
import wfdb
from pathlib import Path
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Paths
BASE = Path('..') / 'data' / 'brugada-huca'
RECORDS_DIR = BASE / 'files'
METADATA_PATH = BASE / 'metadata.csv'

metadata = pd.read_csv(METADATA_PATH)
metadata = metadata[metadata['patient_id'].notna()]
metadata['patient_id'] = metadata['patient_id'].astype(str)

# Bandpass filter helper (common ECG bandpass, 0.5-40 Hz)
def bandpass(signal, fs, low=0.5, high=40.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, signal, axis=0)

# Load a record and return filtered leads (V1/V2/V3 if available)
def load_record(patient_id, leads=('V1', 'V2', 'V3')):
    rec_path = RECORDS_DIR / patient_id / patient_id
    rec = wfdb.rdrecord(str(rec_path))
    sig = rec.p_signal

    lead_idxs = [rec.sig_name.index(l) for l in leads if l in rec.sig_name]
    if len(lead_idxs) == 0:
        lead_idxs = list(range(min(3, sig.shape[1])))

    sig_sel = sig[:, lead_idxs]
    sig_filt = bandpass(sig_sel, rec.fs)
    return sig_filt, rec.fs

# Small feature extractor (per-lead mean/std/ptp/max/min)
def extract_features(signal):
    # signal: (samples, leads)
    feats = []
    for lead_idx in range(signal.shape[1]):
        lead = signal[:, lead_idx]
        feats.extend([
            np.mean(lead),
            np.std(lead),
            np.min(lead),
            np.max(lead),
            np.ptp(lead),
        ])
    return np.array(feats)

In [40]:
# Build dataset from metadata labels
pos_ids = metadata.loc[metadata['brugada'] > 0, 'patient_id'].astype(str).tolist()
neg_ids = metadata.loc[metadata['brugada'] == 0, 'patient_id'].astype(str).tolist()

print(f'Using {len(pos_ids)} positive and {len(neg_ids)} negative records')

def build_dataset(patient_ids, label):
    X = []
    y = []
    for pid in patient_ids:
        try:
            sig, fs = load_record(pid)
            X.append(extract_features(sig))
            y.append(label)
        except Exception as e:
            # Some records may be missing or fail to read; skip them
            print('skipping', pid, '->', str(e))
    return np.stack(X), np.array(y)

X_pos, y_pos = build_dataset(pos_ids, 1)
X_neg, y_neg = build_dataset(neg_ids, 0)

X = np.vstack([X_neg, X_pos])
y = np.concatenate([y_neg, y_pos])

print('Loaded:', X.shape, 'labels:', np.bincount(y))

Using 76 positive and 287 negative records
Loaded: (363, 15) labels: [287  76]


In [42]:
# Train/test split + scaling + SVM
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('Classification report', classification_report(y_test, y_pred))
print('Confusion matrix:', confusion_matrix(y_test, y_pred))

Accuracy: 0.7808
Classification report               precision    recall  f1-score   support

           0       0.92      0.79      0.85        58
           1       0.48      0.73      0.58        15

    accuracy                           0.78        73
   macro avg       0.70      0.76      0.72        73
weighted avg       0.83      0.78      0.80        73

Confusion matrix: [[46 12]
 [ 4 11]]
